# 12 — Regression Precision Under Cognitive Load

**Hypothesis:** High cognitive load during first-pass fixation on the
eventually-clicked result predicts *lower* regression landing precision — because
spatial memory encoding degrades under load.

**Connection to parafoveal preview:** Rayner's parafoveal preview benefit depends
on the useful field of view, which narrows under cognitive load. If load during
initial encoding reduces the spatial context stored around the fixation target,
the motor system has a noisier target representation when executing the return
saccade after a regression. The AFE model's σ² (spatial uncertainty) term should
increase with load.

**Two temporal grains of load measurement:**
1. *Trial-level LHIPA* — wavelet-decomposed pupil signal across the entire trial
   (coarse: averages load over 10-30s)
2. *Encoding-fixation pupil diameter* — mean pupil diameter during the specific
   ~200ms fixation that encodes the target location (fine: load at the exact
   moment of memory formation)


### Key Measures

| Measure | Definition | Units | Interpretation |
|---------|-----------|-------|----------------|
| LHIPA | Low/High Index of Pupillary Activity — ratio of low- to high-frequency pupil signal power (trial-level) | ratio | Lower = higher cognitive load |
| Encoding pupil diameter | Mean pupil diameter (left+right average) during the encoding fixation time window | mm | Larger = higher arousal/load (task-evoked pupillary response) |
| Encoding pupil SD | Standard deviation of pupil diameter during the encoding fixation | mm | Higher = more cognitive instability during encoding |
| Landing offset | \|page_y of regression landing fixation − center_y of clicked result band\| | px | Higher = less precise spatial memory |
| Regression distance | Scroll distance traveled during the regression (backward scroll) | px | Control: longer regressions are inherently less precise |
| Encoding-regression gap | Time between first-pass encoding fixation and regression landing | ms | Control: memory decays over time |


In [ ]:
%matplotlib inline
import csv as _csv
import json, math
import numpy as np
from scipy import stats
from pathlib import Path
from bisect import bisect_right
from collections import defaultdict
import matplotlib.pyplot as plt

from data_loader import (
    get_trial_ids, load_trial, load_lhipa,
    result_band_tops, count_results_html,
    interpolate_scroll, setup_plotting,
)

setup_plotting()

SCREEN_H = 1024
PUPIL_DIR = Path('../AdSERP/data/pupil-data')

# Load LHIPA scores
lhipa_data = load_lhipa()
trial_ids = get_trial_ids()
print(f'Total trials: {len(trial_ids)}')
print(f'Trials with LHIPA: {sum(1 for t in trial_ids if t in lhipa_data)}')
print(f'Pupil files: {len(list(PUPIL_DIR.glob("*.csv")))}')


In [ ]:
# ── Pupil data helpers ──────────────────────────────────────────────────

def load_pupil_samples(trial_id):
    """Load raw pupil diameter samples at 150Hz.
    Returns list of (timestamp_ms, pupil_diameter_mm) tuples.
    Uses mean of left+right when both valid; single eye when one valid."""
    path = PUPIL_DIR / f'{trial_id}.csv'
    if not path.exists():
        return None
    samples = []
    with open(path) as f:
        for row in _csv.DictReader(f):
            try:
                t = float(row['timestamp'])
                lpd, rpd = float(row['LPD']), float(row['RPD'])
                lv, rv = int(row['LPV']), int(row['RPV'])
                if lv and rv:
                    pd = (lpd + rpd) / 2
                elif lv:
                    pd = lpd
                elif rv:
                    pd = rpd
                else:
                    continue
                if pd > 0:
                    samples.append((t, pd))
            except (ValueError, KeyError):
                continue
    return samples if samples else None

def get_pupil_during(samples, t_start, t_end):
    """Mean pupil diameter during a time window."""
    vals = [pd for t, pd in samples if t_start <= t <= t_end]
    return np.mean(vals) if len(vals) >= 3 else None

def get_pupil_sd_during(samples, t_start, t_end):
    """SD of pupil diameter during a time window."""
    vals = [pd for t, pd in samples if t_start <= t <= t_end]
    return np.std(vals) if len(vals) >= 5 else None

def partial_spearman(x, y, covariates):
    """Spearman partial correlation via rank-residualization."""
    rx, ry = stats.rankdata(x), stats.rankdata(y)
    rcovs = np.column_stack([stats.rankdata(c) for c in covariates])
    X_aug = np.column_stack([rcovs, np.ones(len(rx))])
    bx = np.linalg.lstsq(X_aug, rx, rcond=None)[0]
    by = np.linalg.lstsq(X_aug, ry, rcond=None)[0]
    return stats.pearsonr(rx - X_aug @ bx, ry - X_aug @ by)


In [ ]:
# ── Build per-trial measures ────────────────────────────────────────────

records = []
skipped = defaultdict(int)

for i, tid in enumerate(trial_ids):
    if i % 500 == 0:
        print(f'  Processing {i}/{len(trial_ids)}...')

    if tid not in lhipa_data:
        skipped['no_lhipa'] += 1
        continue

    trial = load_trial(tid)
    if trial is None:
        skipped['no_meta'] += 1
        continue

    fixations = trial['fixations']
    clicks = trial['clicks']
    scroll_ts = trial['scroll_ts']
    scroll_ys = trial['scroll_ys']
    doc_h = trial['doc_height']
    scr_h = trial['screen_height']

    if len(fixations) < 5 or not clicks:
        skipped['too_few_fix_or_no_click'] += 1
        continue

    n_results = count_results_html(tid)
    if n_results < 2:
        skipped['too_few_results'] += 1
        continue

    tops = result_band_tops(n_results, doc_h)

    # Click position (last click, page-space Y)
    click_t, click_x, click_y = clicks[-1]
    # Coordinate-safe: click_y is already page-space.
    click_pos = bisect_right(tops, click_y) - 1
    if click_pos < 0 or click_pos >= n_results:
        skipped['click_outside_results'] += 1
        continue

    band_top = tops[click_pos]
    band_bot = tops[click_pos + 1] if click_pos + 1 < n_results else doc_h - 200
    band_center = (band_top + band_bot) / 2

    # ── Assign fixations to positions ────────────────────────────────
    fix_records = []
    for fix in fixations:
        fy_clamped = max(0.0, min(fix['y'], scr_h))  # v5 FPOGY clamp
        scroll_at_fix = interpolate_scroll(fix['t'], scroll_ts, scroll_ys)
        page_y = fy_clamped + scroll_at_fix
        pos = bisect_right(tops, page_y) - 1
        if pos < 0 or pos >= n_results:
            pos = -1
        fix_records.append({
            't': fix['t'], 'page_y': page_y, 'pos': pos,
            'scroll': scroll_at_fix, 'd': fix['d'],
        })

    # ── Detect regressions (scroll-up events) ───────────────────────
    regressions = []
    for j in range(1, len(scroll_ys)):
        if scroll_ys[j] < scroll_ys[j - 1]:
            reg_start_t = scroll_ts[j - 1]
            reg_end_t = scroll_ts[j]
            reg_end_scroll = scroll_ys[j]
            k = j + 1
            while k < len(scroll_ys) and scroll_ys[k] <= scroll_ys[k - 1]:
                reg_end_t = scroll_ts[k]
                reg_end_scroll = scroll_ys[k]
                k += 1
            regressions.append({
                'start_t': reg_start_t, 'end_t': reg_end_t,
                'distance': scroll_ys[j - 1] - reg_end_scroll,
            })

    if not regressions:
        skipped['no_regressions'] += 1
        continue

    # ── First forward-scan fixation on clicked result ────────────────
    encoding_fix = None
    for fr in fix_records:
        if fr['pos'] == click_pos:
            encoding_fix = fr
            break

    if encoding_fix is None:
        skipped['no_encoding_fix'] += 1
        continue

    # ── Extract pupil diameter during encoding fixation ──────────────
    pupil_samples = load_pupil_samples(tid)
    encoding_pd = None
    encoding_pd_sd = None
    if pupil_samples is not None:
        fix_start = encoding_fix['t']
        fix_end = fix_start + encoding_fix['d']
        encoding_pd = get_pupil_during(pupil_samples, fix_start, fix_end)
        encoding_pd_sd = get_pupil_sd_during(pupil_samples, fix_start, fix_end)

    # ── First post-regression fixation on clicked result ─────────────
    landing_fix = None
    landing_reg = None
    for reg in regressions:
        if reg['start_t'] <= encoding_fix['t']:
            continue
        for fr in fix_records:
            if fr['t'] > reg['end_t'] and fr['pos'] == click_pos:
                landing_fix = fr
                landing_reg = reg
                break
        if landing_fix is not None:
            break

    if landing_fix is None:
        skipped['no_landing_on_clicked'] += 1
        continue

    # ── Record ───────────────────────────────────────────────────────
    records.append({
        'trial_id': tid,
        'participant': tid.split('-')[0],
        'lhipa': lhipa_data[tid]['lhipa'],
        'encoding_pd': encoding_pd,
        'encoding_pd_sd': encoding_pd_sd,
        'landing_offset': abs(landing_fix['page_y'] - band_center),
        'time_gap_ms': landing_fix['t'] - encoding_fix['t'],
        'reg_distance': landing_reg['distance'],
        'click_pos': click_pos,
    })

n_pd = sum(1 for r in records if r['encoding_pd'] is not None)
n_sd = sum(1 for r in records if r['encoding_pd_sd'] is not None)
print(f'\nTrials with all measures: {len(records)}')
print(f'  with encoding pupil diameter: {n_pd}')
print(f'  with encoding pupil SD: {n_sd}')
print(f'Skipped: {dict(skipped)}')


In [ ]:
# ── Summary statistics ──────────────────────────────────────────────────

lhipas = np.array([r['lhipa'] for r in records])
offsets = np.array([r['landing_offset'] for r in records])
gaps = np.array([r['time_gap_ms'] for r in records])
distances = np.array([r['reg_distance'] for r in records])

has_pd = [r for r in records if r['encoding_pd'] is not None]
enc_pds = np.array([r['encoding_pd'] for r in has_pd])
enc_offsets = np.array([r['landing_offset'] for r in has_pd])

print(f'N = {len(records)} trials, {len(set(r["participant"] for r in records))} participants')
print(f'N with encoding PD = {len(has_pd)}')
print()
for name, arr in [('LHIPA', lhipas), ('Landing offset (px)', offsets),
                   ('Time gap (ms)', gaps), ('Reg distance (px)', distances),
                   ('Encoding PD (mm)', enc_pds)]:
    print(f'{name:>22s}: median={np.median(arr):.2f}, IQR=[{np.percentile(arr,25):.2f}, {np.percentile(arr,75):.2f}]')


In [ ]:
# ── Trial-level LHIPA × landing offset (coarse grain) ──────────────────

rho_lh, p_lh = stats.spearmanr(lhipas, offsets)
r_lh_ctrl, p_lh_ctrl = partial_spearman(lhipas, offsets, [distances, gaps])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Trial-Level LHIPA × Landing Precision', fontsize=14, fontweight='bold')

# Scatter
ax = axes[0]
ax.scatter(lhipas, offsets, alpha=0.2, s=10, color='#94a3b8', edgecolors='none')
bins = np.linspace(np.percentile(lhipas, 1), np.percentile(lhipas, 99), 15)
bidx = np.digitize(lhipas, bins)
bx, by, bsem = [], [], []
for b in range(1, len(bins)):
    m = bidx == b
    if m.sum() >= 5:
        bx.append(bins[b-1] + (bins[1]-bins[0])/2)
        by.append(np.mean(offsets[m]))
        bsem.append(np.std(offsets[m]) / np.sqrt(m.sum()))
ax.errorbar(bx, by, yerr=bsem, color='#6366f1', lw=2, capsize=3, marker='o', ms=5)
ax.set_xlabel('LHIPA (lower = higher load)')
ax.set_ylabel('Landing offset (px)')
ax.set_title(f'ρ = {rho_lh:.3f}, p = {p_lh:.3f}  |  partial ρ = {r_lh_ctrl:.3f}, p = {p_lh_ctrl:.3f}')

# Terciles
ax = axes[1]
tb = np.percentile(lhipas, [33.3, 66.7])
groups = [offsets[lhipas <= tb[0]], offsets[(lhipas > tb[0]) & (lhipas <= tb[1])], offsets[lhipas > tb[1]]]
means = [np.mean(g) for g in groups]
sems = [np.std(g)/np.sqrt(len(g)) for g in groups]
ns = [len(g) for g in groups]
bars = ax.bar(range(3), means, yerr=sems, capsize=5, color=['#dc2626','#f59e0b','#22c55e'], edgecolor='#333', lw=0.5)
ax.set_xticks(range(3))
ax.set_xticklabels(['High load\n(low LHIPA)', 'Medium', 'Low load\n(high LHIPA)'])
ax.set_ylabel('Mean landing offset (px)')
for bar, n in zip(bars, ns):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, f'n={n}', ha='center', va='bottom', fontsize=9)
h, kw_p = stats.kruskal(*groups)
ax.set_title(f'Kruskal-Wallis H={h:.2f}, p={kw_p:.3f}')

plt.tight_layout()
plt.savefig('../plots-v1/plot_regression_precision_by_load.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Trial-level LHIPA: ρ={rho_lh:.4f} (p={p_lh:.4f}), partial ρ={r_lh_ctrl:.4f} (p={p_lh_ctrl:.4f})')


In [ ]:
# ── Encoding-fixation pupil diameter × landing offset (fine grain) ──────
#
# This is the key test: pupil diameter DURING the encoding fixation
# as a temporally precise load proxy.

enc_gaps = np.array([r['time_gap_ms'] for r in has_pd])
enc_dists = np.array([r['reg_distance'] for r in has_pd])

rho_pd, p_pd = stats.spearmanr(enc_pds, enc_offsets)
r_pd_ctrl, p_pd_ctrl = partial_spearman(enc_pds, enc_offsets, [enc_dists, enc_gaps])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Encoding Pupil Diameter × Landing Precision', fontsize=14, fontweight='bold')

# Scatter
ax = axes[0]
ax.scatter(enc_pds, enc_offsets, alpha=0.25, s=12, color='#6366f1', edgecolors='none')
bins = np.linspace(np.percentile(enc_pds, 2), np.percentile(enc_pds, 98), 15)
bidx = np.digitize(enc_pds, bins)
bx, by, bsem = [], [], []
for b in range(1, len(bins)):
    m = bidx == b
    if m.sum() >= 5:
        bx.append(bins[b-1] + (bins[1]-bins[0])/2)
        by.append(np.mean(enc_offsets[m]))
        bsem.append(np.std(enc_offsets[m]) / np.sqrt(m.sum()))
ax.errorbar(bx, by, yerr=bsem, color='#dc2626', lw=2, capsize=3, marker='o', ms=5, label='Binned mean ± SEM')
ax.set_xlabel('Encoding fixation pupil diameter (mm)')
ax.set_ylabel('Landing offset (px)')
ax.set_title(f'ρ = {rho_pd:.3f}, p = {p_pd:.4f}  |  partial ρ = {r_pd_ctrl:.3f}, p = {p_pd_ctrl:.4f}')
ax.legend(fontsize=9)

# Terciles
ax = axes[1]
tb = np.percentile(enc_pds, [33.3, 66.7])
groups = [enc_offsets[enc_pds <= tb[0]], enc_offsets[(enc_pds > tb[0]) & (enc_pds <= tb[1])], enc_offsets[enc_pds > tb[1]]]
means = [np.mean(g) for g in groups]
sems = [np.std(g)/np.sqrt(len(g)) for g in groups]
ns = [len(g) for g in groups]
bars = ax.bar(range(3), means, yerr=sems, capsize=5, color=['#22c55e','#f59e0b','#dc2626'], edgecolor='#333', lw=0.5)
ax.set_xticks(range(3))
ax.set_xticklabels(['Small pupil\n(low load)', 'Medium', 'Large pupil\n(high load)'])
ax.set_ylabel('Mean landing offset (px)')
for bar, n in zip(bars, ns):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, f'n={n}', ha='center', va='bottom', fontsize=9)
h, kw_p = stats.kruskal(*groups)
ax.set_title(f'Kruskal-Wallis H={h:.2f}, p={kw_p:.3f}')

plt.tight_layout()
plt.savefig('../plots-v1/plot_regression_precision_by_encoding_pd.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Encoding PD: ρ={rho_pd:.4f} (p={p_pd:.6f}), partial ρ={r_pd_ctrl:.4f} (p={p_pd_ctrl:.6f})')
print(f'Tercile means: {[f"{m:.1f}px" for m in means]}')


In [ ]:
# ── Encoding pupil variability (SD) × landing offset ───────────────────
#
# High SD during encoding = more cognitive instability = noisier spatial trace?

has_sd = [r for r in records if r['encoding_pd_sd'] is not None]
print(f'Trials with encoding pupil SD: {len(has_sd)}')

if len(has_sd) >= 30:
    sd_vals = np.array([r['encoding_pd_sd'] for r in has_sd])
    sd_offsets = np.array([r['landing_offset'] for r in has_sd])
    rho_sd, p_sd = stats.spearmanr(sd_vals, sd_offsets)
    sd_gaps = np.array([r['time_gap_ms'] for r in has_sd])
    sd_dists = np.array([r['reg_distance'] for r in has_sd])
    r_sd_ctrl, p_sd_ctrl = partial_spearman(sd_vals, sd_offsets, [sd_dists, sd_gaps])

    print(f'Spearman ρ (encoding PD SD × offset): {rho_sd:.4f}, p = {p_sd:.6f}')
    print(f'Partial ρ (controlling distance + gap): {r_sd_ctrl:.4f}, p = {p_sd_ctrl:.6f}')
else:
    print('Insufficient data for pupil SD analysis')


In [ ]:
# ── Control variables ───────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Control Variables vs Landing Offset', fontsize=14, fontweight='bold')

rho_d, p_d = stats.spearmanr(distances, offsets)
rho_g, p_g = stats.spearmanr(gaps / 1000, offsets)

ax = axes[0]
ax.scatter(distances, offsets, alpha=0.15, s=8, color='#64748b', edgecolors='none')
ax.set_xlabel('Regression distance (px)')
ax.set_ylabel('Landing offset (px)')
ax.set_title(f'ρ = {rho_d:.3f}, p = {p_d:.4f}')

ax = axes[1]
ax.scatter(gaps / 1000, offsets, alpha=0.15, s=8, color='#64748b', edgecolors='none')
ax.set_xlabel('Encoding → regression gap (s)')
ax.set_ylabel('Landing offset (px)')
ax.set_title(f'ρ = {rho_g:.3f}, p = {p_g:.4f}')

plt.tight_layout()
plt.show()

print(f'Regression distance × offset: ρ={rho_d:.4f}, p={p_d:.6f}')
print(f'Time gap × offset:            ρ={rho_g:.4f}, p={p_g:.6f}')


In [ ]:
# ── Per-participant analysis ────────────────────────────────────────────

by_pid = defaultdict(list)
for r in has_pd:
    by_pid[r['participant']].append(r)

pid_medians = []
for pid, trials in by_pid.items():
    if len(trials) < 3:
        continue
    pid_medians.append({
        'pid': pid, 'n': len(trials),
        'median_pd': np.median([t['encoding_pd'] for t in trials]),
        'median_offset': np.median([t['landing_offset'] for t in trials]),
    })

print(f'Participants with ≥3 qualifying trials: {len(pid_medians)}')

if len(pid_medians) >= 5:
    pid_pds = np.array([p['median_pd'] for p in pid_medians])
    pid_offs = np.array([p['median_offset'] for p in pid_medians])
    rho_pid, p_pid = stats.spearmanr(pid_pds, pid_offs)

    fig, ax = plt.subplots(figsize=(8, 6))
    sizes = [p['n'] * 3 for p in pid_medians]
    ax.scatter(pid_pds, pid_offs, s=sizes, alpha=0.6, color='#6366f1', edgecolors='#333', lw=0.5)
    ax.set_xlabel('Median encoding pupil diameter (mm)')
    ax.set_ylabel('Median landing offset (px)')
    ax.set_title(f'Per-Participant Medians (ρ = {rho_pid:.3f}, p = {p_pid:.4f})')
    for n_ex in [3, 10, 20]:
        ax.scatter([], [], s=n_ex*3, color='#6366f1', alpha=0.6, label=f'{n_ex} trials')
    ax.legend(title='Trial count', loc='upper right', fontsize=9)
    plt.tight_layout()
    plt.show()
    print(f'Per-participant ρ = {rho_pid:.4f}, p = {p_pid:.6f}')


## Interpretation

### Trial-level LHIPA (coarse temporal grain)
Trial-level LHIPA averages cognitive load across the entire trial (10-30s),
diluting signal from the critical encoding moment. Expected to be weak or null.

### Per-fixation pupil diameter (fine temporal grain)
Pupil diameter during the encoding fixation captures load at the exact moment
spatial memory is formed (~200ms window at 150Hz ≈ 30 samples).

**If significant (positive ρ, larger pupil → worse precision):**
The useful field of view narrows under momentary load, degrading spatial encoding.
*Implication for Scrutinizer:* foveation radius should shrink when instantaneous
pupil diameter is elevated — load-modulated mask.

**If null even at fixation level:**
Spatial memory for SERP result positions is robust to normal cognitive load
variation during browsing. The motor system may rely on non-pupillometric cues
(proprioceptive scroll memory, visual landmarks) rather than purely spatial memory.

### Methodological notes
- Pupil diameter is confounded with luminance — SERP pages have uniform backgrounds
  but result content (images vs text) could drive local differences
- The encoding fixation is the *first* fixation on the clicked result; if encoding
  happens across multiple fixations, a single-fixation window is conservative
- FPOGD in fixation CSVs is fixation *duration* (ms), not pupil diameter — the
  pupil signal comes from the separate pupil-data CSVs at 150Hz
